In [1]:
from silero_vad.utils_vad import init_jit_model, OnnxWrapper
import torch
from torch import nn
loaded = torch.jit.load("/home/duy/miniconda3/envs/bnlu/lib/python3.13/site-packages/silero_vad/data/silero_vad.jit")

In [19]:
def get_buffer_and_param(module: nn.Module, prefix: str = ""):
    """Recursively get all buffer and paramters"""
    def _shape(x):
        try:
            return tuple(x.shape)
        except Exception:
            return type(x)

    # parameters and buffers for this module only
    for name, param in module.named_parameters(recurse=False):
        print(f"[params] {prefix}{name}: {_shape(param)}")

    for name, buf in module.named_buffers(recurse=False):
        print(f"[buffer] {prefix}{name}: {_shape(buf)}")

    # recurse into children
    for child_name, child in module.named_children():
        get_buffer_and_param(child, prefix + child_name + ".")
    

get_buffer_and_param(loaded)

[buffer] _model.stft.forward_basis_buffer: (258, 1, 256)
[params] _model.encoder.0.reparam_conv.weight: (128, 129, 3)
[params] _model.encoder.0.reparam_conv.bias: (128,)
[params] _model.encoder.1.reparam_conv.weight: (64, 128, 3)
[params] _model.encoder.1.reparam_conv.bias: (64,)
[params] _model.encoder.2.reparam_conv.weight: (64, 64, 3)
[params] _model.encoder.2.reparam_conv.bias: (64,)
[params] _model.encoder.3.reparam_conv.weight: (128, 64, 3)
[params] _model.encoder.3.reparam_conv.bias: (128,)
[params] _model.decoder.rnn.weight_ih: (512, 128)
[params] _model.decoder.rnn.weight_hh: (512, 128)
[params] _model.decoder.rnn.bias_ih: (512,)
[params] _model.decoder.rnn.bias_hh: (512,)
[params] _model.decoder.decoder.2.weight: (1, 128, 1)
[params] _model.decoder.decoder.2.bias: (1,)
[buffer] _model_8k.stft.forward_basis_buffer: (130, 1, 128)
[params] _model_8k.encoder.0.reparam_conv.weight: (128, 65, 3)
[params] _model_8k.encoder.0.reparam_conv.bias: (128,)
[params] _model_8k.encoder.1.rep

# Overview the architectural reversed engineer process
To success at the reverse engineer, 


The module list of SileroVAD is as blow:

```
vadrnn
 stft: STFT(...)
 encoder
   [{
     - se: Identity
     - activation: ReLU
     - reparam_conv: Conv1d(...)
   } x 4]
   
   
   
 decoder
   - decoder: Seq(Dropout -> Relu -> Conv1d(...) -> Sigmoid)
   - rnn: LSTMCell
```

First, we need to reverse engineer the STFT part

# Reversed engineer the STFT

In [4]:
module = getattr(loaded._model, "stft")

## Reverse the buffer (tricky)

This module has single `forward_basis_buffer` to compute STFT coefficents.


In [6]:
for name, params in module.named_parameters():
    print(f"{name=}: {params.shape}")

obtained_filter = module.forward_basis_buffer

print(module.transform_.code)

def transform_(self,
    input_data: Tensor) -> Tuple[Tensor, Tensor]:
  padding = self.padding
  input_data0 = torch.unsqueeze((padding).forward(input_data, ), 1)
  forward_basis_buffer = self.forward_basis_buffer
  hop_length = self.hop_length
  forward_transform = torch.conv1d(input_data0, forward_basis_buffer, None, [hop_length], [0])
  filter_length = self.filter_length
  _0 = torch.add(torch.div(filter_length, 2), 1)
  cutoff = int(_0)
  _1 = torch.slice(torch.slice(forward_transform), 1, None, cutoff)
  real_part = torch.to(torch.slice(_1, 2), 6)
  _2 = torch.slice(torch.slice(forward_transform), 1, cutoff)
  imag_part = torch.to(torch.slice(_2, 2), 6)
  _3 = torch.add(torch.pow(real_part, 2), torch.pow(imag_part, 2))
  magnitude = torch.sqrt(_3)
  phase = torch.atan2(ops.prim.data(imag_part), ops.prim.data(real_part))
  return (magnitude, phase)



In [ ]:
# TODO: Guess the basis creation, this is critical, numerical error on this part can cause wrong prediction!

def create_stft_basis(n_fft: int, win_length: int, hop_length: int) -> torch.Tensor:
    """
    Create STFT basis buffer programmatically.
    
    For 16kHz: n_fft=512, resulting in 258 frequency bins (512//2 + 1)
    Shape: [freq_bins, 1, n_fft]
    
    Args:
        n_fft: FFT size
        win_length: Window length
        hop_length: Hop length between frames
        
    Returns:
        forward_basis_buffer: Shape [n_fft//2 + 1, 1, n_fft]
    """
    # Create Hann window
    window = torch.hann_window(win_length, periodic=False)
    
    # Create basis matrix using STFT
    # STFT returns complex tensor, we take both real and imaginary parts
    basis = torch.stft(
        torch.eye(n_fft),  # Identity matrix as input
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        return_complex=True
    )
    
    # Shape: [n_fft, freq_bins, time_steps] -> reshape to [freq_bins, 1, n_fft]
    basis = basis.squeeze(-1).unsqueeze(1)  # [freq_bins, 1, n_fft]
    
    return basis

basis = create_stft_basis(n_fft=256, win_length=256, hop_length=256)

basis.shape

In [ ]:
# Im curious about 



# Reverse engineer Encoder

In [15]:
encoder = getattr(loaded._model, "encoder")
for name, buffer in encoder.named_buffers():
    print(f"{name}: {buffer.shape}")
    
for name, param in encoder.named_parameters():
    print(f"{name}: {param.shape}")

0.reparam_conv.weight: torch.Size([128, 129, 3])
0.reparam_conv.bias: torch.Size([128])
1.reparam_conv.weight: torch.Size([64, 128, 3])
1.reparam_conv.bias: torch.Size([64])
2.reparam_conv.weight: torch.Size([64, 64, 3])
2.reparam_conv.bias: torch.Size([64])
3.reparam_conv.weight: torch.Size([128, 64, 3])
3.reparam_conv.bias: torch.Size([128])


There is no hidden buffer!

The progress so far

VADRNN -> stft, encoder (SileroVADBlock, ...)

Only know the Conv1d params of each `reparam_conv` and we succefully reverse encoder architecture~ 

In [41]:


block_0 = getattr(encoder, "0")

print(block_0)

print("-- The forward code")
print(block_0.code)


RecursiveScriptModule(
  original_name=SileroVadBlock
  (se): RecursiveScriptModule(original_name=Identity)
  (activation): RecursiveScriptModule(original_name=ReLU)
  (reparam_conv): RecursiveScriptModule(original_name=Conv1d)
)
-- The forward code
def forward(self,
    x: Tensor) -> Tensor:
  activation = self.activation
  se = self.se
  reparam_conv = self.reparam_conv
  _0 = (se).forward((reparam_conv).forward(x, ), )
  return (activation).forward(_0, )



Okay, it's just weight and bias. Encoder is piece of cake!

In [53]:
for i in range(4):
    print(f"The {i} block")
    block = getattr(encoder, str(i))
    
    print(f"{block.reparam_conv.in_channels=}")
    print(f"{block.reparam_conv.out_channels=}")
    print(f"{block.reparam_conv.kernel_size=}") # 3
    print(f"{block.reparam_conv.padding=}") # 1
    #  -> padding=same
    # They are just default value
    print(f"{block.reparam_conv.dilation=}")
    print(f"{block.reparam_conv.groups=}")
    print(f"{block.reparam_conv.stride=}")
    print(f"{block.reparam_conv.bias=}" is None)
    
    print("---")

The 0 block
block.reparam_conv.in_channels=129
block.reparam_conv.out_channels=128
block.reparam_conv.kernel_size=(3,)
block.reparam_conv.padding=(1,)
block.reparam_conv.dilation=(1,)
block.reparam_conv.groups=1
block.reparam_conv.stride=(1,)
False
---
The 1 block
block.reparam_conv.in_channels=128
block.reparam_conv.out_channels=64
block.reparam_conv.kernel_size=(3,)
block.reparam_conv.padding=(1,)
block.reparam_conv.dilation=(1,)
block.reparam_conv.groups=1
block.reparam_conv.stride=(2,)
False
---
The 2 block
block.reparam_conv.in_channels=64
block.reparam_conv.out_channels=64
block.reparam_conv.kernel_size=(3,)
block.reparam_conv.padding=(1,)
block.reparam_conv.dilation=(1,)
block.reparam_conv.groups=1
block.reparam_conv.stride=(2,)
False
---
The 3 block
block.reparam_conv.in_channels=64
block.reparam_conv.out_channels=128
block.reparam_conv.kernel_size=(3,)
block.reparam_conv.padding=(1,)
block.reparam_conv.dilation=(1,)
block.reparam_conv.groups=1
block.reparam_conv.stride=(1,)
Fa

This follows bottleneck module. There is 4 blocks, both of which use padding=same strategy


-> Done. We can code the init of encoder part now.

# Reversed engineer the decoder

In [55]:
decoder = getattr(loaded._model, "decoder")
decoder

RecursiveScriptModule(
  original_name=VADDecoderRNNJIT
  (rnn): RecursiveScriptModule(original_name=LSTMCell)
  (decoder): RecursiveScriptModule(
    original_name=Sequential
    (0): RecursiveScriptModule(original_name=Dropout)
    (1): RecursiveScriptModule(original_name=ReLU)
    (2): RecursiveScriptModule(original_name=Conv1d)
    (3): RecursiveScriptModule(original_name=Sigmoid)
  )
)

-> What to reverse: The parameters LSTMCell and Conv1d. 

In [63]:
# RNN
print(decoder.rnn.input_size)
print(decoder.rnn.hidden_size)
# -> Done

128
128


In [66]:
decoder.decoder

RecursiveScriptModule(
  original_name=Sequential
  (0): RecursiveScriptModule(original_name=Dropout)
  (1): RecursiveScriptModule(original_name=ReLU)
  (2): RecursiveScriptModule(original_name=Conv1d)
  (3): RecursiveScriptModule(original_name=Sigmoid)
)

Just a single block

In [69]:
# Decoder/Conv1d

conv = getattr(decoder.decoder, "2")
print(f"{conv.in_channels=}")
print(f"{conv.out_channels=}")
print(f"{conv.kernel_size=}") # 3
print(f"{conv.padding=}") # 1
#  -> padding=same
# They are just default value
print(f"{conv.dilation=}")
print(f"{conv.groups=}")
print(f"{conv.stride=}")
print(f"{conv.bias=}" is None)

conv.in_channels=128
conv.out_channels=1
conv.kernel_size=(1,)
conv.padding=(0,)
conv.dilation=(1,)
conv.groups=1
conv.stride=(1,)
False


In [70]:
print(decoder.code)

def forward(self,
    x: Tensor,
    state: Tensor=CONSTANTS.c0) -> Tuple[Tensor, Tensor]:
  x0 = torch.squeeze(x, -1)
  if bool(torch.len(state)):
    rnn = self.rnn
    _0 = (torch.select(state, 0, 0), torch.select(state, 0, 1))
    h0, c0, = (rnn).forward(x0, _0, )
    h, c = h0, c0
  else:
    rnn0 = self.rnn
    h1, c1, = (rnn0).forward(x0, None, )
    h, c = h1, c1
  x1 = torch.to(torch.unsqueeze(h, -1), 6)
  state0 = torch.stack([h, c])
  decoder = self.decoder
  x2 = (decoder).forward(x1, )
  return (x2, state0)



# Put all together: reversed engineer the model

In [71]:
print(loaded.code)

def forward(self,
    x: Tensor,
    sr: int) -> Tensor:
  _0 = "Provided number of samples is {} (Supported values: 256 for 8000 sample rate, 512 for 16000)"
  _1 = uninitialized(Tensor)
  x0, sr0, = (self)._validate_input(x, sr, )
  if torch.eq(sr0, 16000):
    num_samples = 512
  else:
    num_samples = 256
  _2 = torch.ne((torch.size(x0))[-1], num_samples)
  if _2:
    _3 = torch.format(_0, (torch.size(x0))[-1])
    ops.prim.RaiseException(_3, "builtins.ValueError")
  else:
    pass
  batch_size = (torch.size(x0))[0]
  if torch.eq(sr0, 16000):
    _model = self._model
    context_size = _model.context_size_samples
  else:
    _model_8k = self._model_8k
    context_size = _model_8k.context_size_samples
  _last_sr = self._last_sr
  if bool(_last_sr):
    _last_sr0 = self._last_sr
    _4 = torch.ne(_last_sr0, sr0)
  else:
    _4 = False
  if _4:
    _5 = (self).reset_states()
  else:
    pass
  _last_batch_size = self._last_batch_size
  if bool(_last_batch_size):
    _last_batch_size0

In [74]:
print(loaded._validate_input.code)

def _validate_input(self,
    x: Tensor,
    sr: int) -> Tuple[Tensor, int]:
  _0 = "Too many dimensions for input audio chunk {}"
  _1 = "Supported sampling rates: {} (or multiply of 16000)"
  if torch.eq(torch.dim(x), 1):
    x2 = torch.unsqueeze(x, 0)
  else:
    x2 = x
  if torch.gt(torch.dim(x2), 2):
    ops.prim.RaiseException(torch.format(_0, torch.dim(x2)), "builtins.ValueError")
  else:
    pass
  if torch.ne(sr, 16000):
    _3 = torch.eq(torch.remainder(sr, 16000), 0)
    _2 = _3
  else:
    _2 = False
  if _2:
    step = torch.floordiv(sr, 16000)
    x4 = torch.slice(torch.slice(x2), 1, None, None, step)
    sr1, x3 = 16000, x4
  else:
    sr1, x3 = sr, x2
  sample_rates = self.sample_rates
  _4 = torch.__not__(torch.__contains__(sample_rates, sr1))
  if _4:
    sample_rates0 = self.sample_rates
    ops.prim.RaiseException(torch.format(_1, sample_rates0), "builtins.ValueError")
  else:
    pass
  _5 = torch.gt(torch.div(sr1, (torch.size(x3))[1]), 31.25)
  if _5:
    ops.prim

In [72]:
print(loaded.reset_states.code)

def reset_states(self) -> NoneType:
  self._state = torch.zeros([0])
  self._context = torch.zeros([0])
  self._last_sr = 0
  self._last_batch_size = 0
  return None



## Reversed engineer the vadrnn

In [87]:
t = torch.rand((1, 1, 512))
tr = torch.conv1d(t, weight=obtained_filter, stride=[128], padding=[0])
tr.shape

torch.Size([1, 258, 3])

In [93]:
print(loaded._model.code)

def forward(self,
    x: Tensor,
    state: Tensor=CONSTANTS.c0) -> Tuple[Tensor, Tensor]:
  x0 = (self).run_extractors(x, )
  encoder = self.encoder
  x1 = (encoder).forward(x0, )
  decoder = self.decoder
  x2, state0, = (decoder).forward(x1, state, )
  out = torch.unsqueeze(torch.mean(torch.squeeze(x2, 1), [1]), 1)
  return (out, state0)



In [116]:
print(loaded._model.stft.padding.code)

def forward(self,
    input: Tensor) -> Tensor:
  _0 = __torch__.torch.nn.functional.___torch_mangle_14.pad
  return _0(input, [0, 64], "reflect", None, )



In [109]:
for name, p in loaded._model.stft.named_buffers():
    print(name)

forward_basis_buffer


In [122]:
sum(p.numel() for _,p in loaded._model.named_parameters())

243585

# Check the process

In [1]:
import torch

from rveng.model import VAD, VADRNN

vad = VAD()
vad._model = VADRNN()

loaded = torch.jit.load("/home/duy/miniconda3/envs/bnlu/lib/python3.13/site-packages/silero_vad/data/silero_vad.jit")


In [2]:
for n, p in vad._model.named_parameters():
    print(n)

encoder.0.reparam_conv.weight
encoder.0.reparam_conv.bias
encoder.1.reparam_conv.weight
encoder.1.reparam_conv.bias
encoder.2.reparam_conv.weight
encoder.2.reparam_conv.bias
encoder.3.reparam_conv.weight
encoder.3.reparam_conv.bias
decoder.decoder.2.weight
decoder.decoder.2.bias
decoder.rnn.weight_ih
decoder.rnn.weight_hh
decoder.rnn.bias_ih
decoder.rnn.bias_hh


In [3]:
vad._model.load_state_dict(loaded._model.state_dict())


<All keys matched successfully>

In [4]:
((vad._model.stft.forward_basis_buffer - loaded._model.stft.forward_basis_buffer) ** 2).max()

tensor(0.)

In [11]:
import torch
from rveng.model import VAD, VADRNN
from collections import OrderedDict

state_dict = OrderedDict()

class Validator:
    def __init__(self):
        self.vad = VAD()
        self.vad._model = VADRNN()

        self.loaded = torch.jit.load("assets/silero_vad.jit")
            
        self.vad._model.load_state_dict(self.loaded._model.state_dict())

        self.loaded._model.eval()
        self.vad._model.eval()

    def reset_states(self):
        self.loaded.reset_states()
        self.vad.reset_states()

    def compute_error(self):
        t = torch.rand((1, 512))
        sr = 16000
        out_jit = self.loaded(t, sr=sr)
        out_rv = self.vad(t, sr=sr)
        
        print(out_jit)
        print(out_rv)
        
    def compute_error_encoder(self):
        t = torch.rand((1, 129, 1))
        
        out_orig = self.loaded._model.encoder(t)
        # print(out_orig)
        out_reversed = self.vad._model.encoder(t)
        # print(out_reversed)
        
        err = ((out_orig - out_reversed)**2).mean()
        return err
        
    def compute_error_decoder_single_chunk(self):
        t = torch.rand((1, 128))
        
        out_orig, state_out_orig = self.loaded._model.decoder(t)
        out_reversed, state_out_reversed = self.vad._model.decoder(t)
        
        err_out = ((out_orig - out_reversed)**2).mean()
        err_state = ((state_out_orig - state_out_reversed)**2).mean()
        
        return err_out, err_state
    
    def compute_error_decoder_sequence(self, sequence_length=100):
        err_outs = []
        err_states = []
        
        state_orig = None
        state_reversed = None
        
        for i in range(sequence_length):
            t = torch.rand((1, 128))
            out_orig, state_orig = self.loaded._model.decoder(t, state_orig)
            out_reversed, state_reversed = self.vad._model.decoder(t, state_reversed)
            
            err_out = ((out_orig - out_reversed)**2).mean()
            err_state = ((state_orig - state_reversed)**2).mean()
            
            
            err_outs.append(err_out)
            err_states.append(err_state)
            
        err_outs = torch.stack(err_outs)    
        err_states = torch.stack(err_states)
        
        return err_outs.mean(), err_states.mean()
    
    def compute_error_prenet(self):
        t = torch.rand((1, 4096))
        orig = self.loaded._model.stft(t)
        reversed = self.vad._model.stft(t)
        
        err = ((orig - reversed)**2).mean()
        
        return err
        
v = Validator()
# v.compute_error()
encoder_err = v.compute_error_encoder()
print(f"{encoder_err=}")

v.reset_states()
decoder_err, state_err = v.compute_error_decoder_single_chunk()
print(f"{decoder_err=}")
print(f"{state_err=}")

v.reset_states()
decoder_err_sequence, state_err_sequence = v.compute_error_decoder_sequence()
print(f"{decoder_err_sequence=}")
print(f"{state_err_sequence=}")


v.reset_states()
mag_err = v.compute_error_prenet()
print(f"{mag_err=}")


v.reset_states()
v.compute_error()

encoder_err=tensor(0., grad_fn=<MeanBackward0>)
decoder_err=tensor(0., grad_fn=<MeanBackward0>)
state_err=tensor(0., grad_fn=<MeanBackward0>)
decoder_err_sequence=tensor(0., grad_fn=<MeanBackward0>)
state_err_sequence=tensor(0., grad_fn=<MeanBackward0>)
mag_err=tensor(0.)
tensor([[0.0061]], grad_fn=<UnsqueezeBackward0>)
tensor([[[0.0006]]], grad_fn=<MeanBackward1>)


In [19]:
print(loaded._model.stft.code)

def forward(self,
    input_data: Tensor) -> Tensor:
  return ((self).transform_(input_data, ))[0]

